# Amended Stage 11C — Original-First Official-Route Recovery and Outcome-Free Reserve Screening

This notebook implements the sealed Protocol Amendment A1. It first attempts the five original breast-ultrasound development datasets in their frozen Stage 10 order, using only the official routes authorised by A1. Mendeley `Download All` endpoints, Zenodo records/files, the TCIA NBIA v4 production API, official clients or publisher/author repositories, and checksum-audited manual official receipts are allowed. Unofficial mirrors remain prohibited.

Only after all five originals are adjudicated may the three frozen reserves be screened in their fixed order. Reserve activation uses acquisition and governance evidence only: official identity, rights, pathology-compatible labels, patient/lesion grouping, integrity, and provenance/duplicate risk. Embeddings, source AUC, transfer performance, calibration, DDO2 and blind outcomes are forbidden activation inputs.

The notebook seals a pre-fit development roster and an exact Stage 11D handoff. It does **not** compute embeddings, fit a source axis, evaluate a transfer edge, fit DDO2, enter Stage 12, access a locked-blind asset, or perform blind validation. Any incomplete official route or governance item becomes a typed `HOLD`; it never becomes a model-performance failure.


In [1]:
# @title 11C-0. Mount Drive, verify A1 lineage, and seal the pre-request protocol
import hashlib
import io
import json
import math
import os
import re
import shutil
import tarfile
import time
import warnings
import zipfile
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urlparse

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print

IN_COLAB = False
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except Exception:
    pass

TEST_MODE = bool(os.environ.get("CDO_STAGE11C_TEST_MODE") == "1" and not IN_COLAB)
DEFAULT_ROOT = Path("/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability") if IN_COLAB else Path("/tmp/Cross-Modal_Diagnostic_Observability")
PROJECT_ROOT = Path(os.environ.get("CDO_PROJECT_ROOT", str(DEFAULT_ROOT)))
CODE_ROOT = PROJECT_ROOT / "05_Code" / "Cross_Modal"
CROSS_MODAL_ROOT = PROJECT_ROOT / "06_Data_Records" / "Cross_Modal"
A1_ROOT = CROSS_MODAL_ROOT / "Protocol_Amendment_A1_Official_Route_Recovery_And_Development_Only_Substitution_Rules_v0.1"
STAGE11C_ROOT = CROSS_MODAL_ROOT / "Stage11C_Amended_Original_First_Official_Route_Recovery_And_Outcome_Free_Reserve_Screening_v0.1"

PROTOCOL_ROOT = STAGE11C_ROOT / "00_Protocol"
RECOVERY_ROOT = STAGE11C_ROOT / "01_Original_Official_Route_Recovery"
RESERVE_ROOT = STAGE11C_ROOT / "02_Outcome_Free_Reserve_Screening"
ROSTER_ROOT = STAGE11C_ROOT / "03_Final_Development_Roster"
FIREWALL_ROOT = STAGE11C_ROOT / "04_Firewall"
RESULT_ROOT = STAGE11C_ROOT / "05_Results"
for directory in [CODE_ROOT, PROTOCOL_ROOT, RECOVERY_ROOT, RESERVE_ROOT, ROSTER_ROOT, FIREWALL_ROOT, RESULT_ROOT]:
    directory.mkdir(parents=True, exist_ok=True)

NOTEBOOK_NAME = "CrossModal_Stage11C_Amended_Original_First_Official_Route_Recovery_And_Outcome_Free_Reserve_Screening_v0.1.ipynb"
NOTEBOOK_PATH = CODE_ROOT / NOTEBOOK_NAME
TEMP_ROOT = Path(os.environ.get("CDO_STAGE11C_TEMP_ROOT", "/content/stage11c_tmp" if IN_COLAB else "/tmp/stage11c_tmp"))
TEMP_ROOT.mkdir(parents=True, exist_ok=True)

A1_SEAL_PATH = A1_ROOT / "00_Protocol" / "Protocol_Amendment_A1_Seal_v0.1.json"
A1_ROUTE_PATH = A1_ROOT / "02_Official_Route_Recovery" / "A1_Official_Route_Recovery_Registry_v0.1.csv"
A1_MANUAL_SCHEMA_PATH = A1_ROOT / "02_Official_Route_Recovery" / "A1_Manual_Official_Download_Receipt_Schema_v0.1.csv"
A1_RESERVE_PATH = A1_ROOT / "03_Development_Reserve" / "A1_Frozen_Development_Reserve_Pool_v0.1.csv"
A1_EXCLUSION_PATH = A1_ROOT / "03_Development_Reserve" / "A1_Explicit_Exclusion_Registry_v0.1.csv"
A1_ACTIVATION_PATH = A1_ROOT / "03_Development_Reserve" / "A1_Outcome_Free_Activation_Policy_v0.1.json"
A1_BLIND_PATH = A1_ROOT / "04_Firewall" / "A1_Locked_Blind_Continuity_Registry_v0.1.csv"
A1_FINAL_PATH = A1_ROOT / "05_Results" / "Protocol_Amendment_A1_Complete_v0.1.json"

PROTOCOL_SEAL_PATH = PROTOCOL_ROOT / "Stage11C_Amended_Recovery_And_Reserve_Screening_Protocol_Seal_v0.1.json"
PARENT_COMMITMENT_PATH = PROTOCOL_ROOT / "Stage11C_A1_Parent_Input_Commitment_v0.1.csv"
REQUEST_LOG_PATH = RECOVERY_ROOT / "Stage11C_Official_Request_Audit_v0.1.csv"
FILE_RECEIPT_PATH = RECOVERY_ROOT / "Stage11C_Official_File_Receipt_Ledger_v0.1.csv"
ORIGINAL_OUTCOME_PATH = RECOVERY_ROOT / "Stage11C_Original_Recovery_Outcome_v0.1.csv"
RESERVE_EVIDENCE_PATH = RESERVE_ROOT / "Stage11C_Reserve_Screening_Evidence_v0.1.csv"
RESERVE_DECISION_PATH = RESERVE_ROOT / "Stage11C_Reserve_Activation_Decision_v0.1.csv"
ROSTER_PATH = ROSTER_ROOT / "Stage11C_Final_Development_Roster_v0.1.csv"
HANDOFF_PATH = ROSTER_ROOT / "Stage11C_Stage11D_Prefit_Handoff_v0.1.json"
BLIND_CONTINUITY_PATH = FIREWALL_ROOT / "Stage11C_Locked_Blind_Continuity_v0.1.csv"
VALIDITY_PATH = FIREWALL_ROOT / "Stage11C_Independent_Validity_Checks_v0.1.csv"
REPORT_PATH = RESULT_ROOT / "Stage11C_Amended_Recovery_And_Reserve_Screening_Report_v0.1.md"
FIGURE_PATH = RESULT_ROOT / "Stage11C_Original_And_Reserve_Status_v0.1.png"
OUTPUT_MANIFEST_PATH = RESULT_ROOT / "Stage11C_Output_Integrity_Manifest_v0.1.csv"
RUNTIME_STATE_PATH = RESULT_ROOT / "Stage11C_Runtime_State_v0.1.json"
FINAL_RECORD_PATH = RESULT_ROOT / "Stage11C_Amended_Recovery_And_Reserve_Screening_Complete_v0.1.json"

EXPECTED_A1_SEAL = "bb85f49c2378600a50619624ad322dc824d662d8eadb73fcf1d5532ee067082a"
EXPECTED_A1_ROUTE = "baecfc013349037d739b5058e45938e0f785718465bd88d45d58de5f3f4b7399"
EXPECTED_A1_RESERVE = "37c1d5da4cfd2057ee3438cacfbc831df6d4cdcb5c7efd87cf629d8cf871e912"
EXPECTED_A1_ACTIVATION = "fd9dc25be5699c4dee17dc7af18f1bb9e063bafc6d7835087212e113ead0ce6a"
EXPECTED_A1_FINAL = "4003be92df8a9c84352062383e8a94d066cc0e2bb2978cfb1af8968e51b85320"
ORIGINAL_IDS = ["BUS_BRA_2024", "BUSI_WHU_2025_V3", "BREAST_LESIONS_USG_2024", "BUS_UCLM_2025_V3", "RODRIGUES_BUI_2017"]
RESERVE_IDS = ["UDIAT_B_2017", "MICCAI_BUV_2022", "UVBLS200_2025"]
LOCKED_BLIND_IDS = ["BUSI_CAIRO_2019", "OASBUD_2017", "DERM7PT_2019"]
TARGET_ROSTER = 5
MINIMUM_FEASIBLE_DOMAINS = 4
MAXIMUM_NEW_STAGE11C_BYTES = 256 * 1024 * 1024

def utc_now():
    return datetime.now(timezone.utc).isoformat()

def sha256_bytes(raw):
    return hashlib.sha256(raw).hexdigest()

def sha256_file(path, block_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while True:
            block = handle.read(block_size)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()

def sha256_json(payload):
    raw = json.dumps(payload, sort_keys=True, separators=(",", ":"), ensure_ascii=False).encode("utf-8")
    return hashlib.sha256(raw).hexdigest()

def canonical_csv_text(frame):
    return frame.to_csv(index=False, lineterminator="\n", float_format="%.12g")

def write_immutable_text(path, text):
    path = Path(path)
    if path.is_file():
        assert path.read_text(encoding="utf-8") == text, f"Existing immutable output differs: {path}"
    else:
        path.write_text(text, encoding="utf-8")

def write_immutable_csv(path, frame):
    write_immutable_text(path, canonical_csv_text(frame))

def write_immutable_json(path, payload):
    write_immutable_text(path, json.dumps(payload, indent=2, ensure_ascii=False) + "\n")

def atomic_json(path, payload):
    temporary = Path(str(path) + ".tmp")
    temporary.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
    os.replace(temporary, path)

def verify_self_hashed_json(path, hash_field, expected=None):
    payload = json.loads(Path(path).read_text(encoding="utf-8"))
    claim = payload[hash_field]
    reduced = dict(payload); reduced.pop(hash_field)
    assert sha256_json(reduced) == claim, f"Self-hash mismatch: {path}"
    if expected is not None:
        assert claim == expected, f"Unexpected frozen claim: {path}"
    return payload

def normalised_notebook_source_sha256(path):
    notebook = json.loads(Path(path).read_text(encoding="utf-8"))
    payload = []
    for cell in notebook.get("cells", []):
        if cell.get("cell_type") in {"code", "markdown"}:
            source = cell.get("source", [])
            source = "".join(source) if isinstance(source, list) else str(source)
            payload.append({"cell_type": cell["cell_type"], "source": source.replace("\r\n", "\n")})
    return sha256_json(payload)

def create_test_a1_fixture():
    if A1_FINAL_PATH.is_file():
        return
    for p in [A1_SEAL_PATH, A1_ROUTE_PATH, A1_MANUAL_SCHEMA_PATH, A1_RESERVE_PATH, A1_EXCLUSION_PATH, A1_ACTIVATION_PATH, A1_BLIND_PATH, A1_FINAL_PATH]:
        p.parent.mkdir(parents=True, exist_ok=True)
    route = pd.DataFrame([
        {"dataset_id": d, "original_priority": i + 1, "original_role_retained": "DEVELOPMENT_EXTENSION", "official_landing_url": u, "permitted_route_order": "OFFICIAL_ONLY", "permitted_hosts": h, "required_identity_gate": "OFFICIAL_IDENTITY", "required_governance_gate": "RIGHTS;LABEL;GROUP;INTEGRITY;DUPLICATE"}
        for i, (d, u, h) in enumerate([
            (ORIGINAL_IDS[0], "https://zenodo.org/records/8231412", "zenodo.org|files.zenodo.org|github.com|raw.githubusercontent.com"),
            (ORIGINAL_IDS[1], "https://data.mendeley.com/datasets/k6cpmwybk3/3", "data.mendeley.com"),
            (ORIGINAL_IDS[2], "https://www.cancerimagingarchive.net/collection/breast-lesions-usg/", "cancerimagingarchive.net|www.cancerimagingarchive.net|nbia.cancerimagingarchive.net"),
            (ORIGINAL_IDS[3], "https://data.mendeley.com/datasets/7fvgj4jsp7/3", "data.mendeley.com"),
            (ORIGINAL_IDS[4], "https://data.mendeley.com/datasets/wmy84gzngw/1", "data.mendeley.com"),
        ])
    ])
    reserve = pd.DataFrame([{"reserve_priority": i + 1, "dataset_id": d, "role_if_activated": "DEVELOPMENT_EXTENSION_A1_RESERVE", "official_primary_record": "https://example.invalid/official", "mandatory_pre_activation_gates": "RIGHTS|LABEL|GROUP|INTEGRITY|DUPLICATE", "status": "CONDITIONAL_RESERVE_NOT_ACCESSED_NOT_ACTIVATED"} for i, d in enumerate(RESERVE_IDS)])
    blind = pd.DataFrame([{"dataset_id": d, "a1_metadata_access_permitted": False, "a1_image_access_permitted": False, "a1_label_access_permitted": False, "a1_development_role_permitted": False, "a1_reserve_substitute_permitted": False, "a1_status": "UNCHANGED_PERMANENTLY_LOCKED_BLIND"} for d in LOCKED_BLIND_IDS])
    activation = {"target_development_domain_count": 5, "minimum_qualified_domains_with_any_count_gate_possibility": 4, "additional_edges_needed": 9, "activation_inputs_permitted": ["official availability", "licence and use terms", "released label semantics", "patient or lesion grouping", "file integrity", "development-only provenance and duplicate evidence"], "activation_inputs_prohibited": ["embedding geometry", "source AUC", "source recoverability", "transfer performance", "calibration", "operating point", "DDO2", "blind outcome"], "post_roster_rule": "source-gate failure never activates another reserve"}
    activation["policy_sha256"] = sha256_json(activation)
    write_immutable_csv(A1_ROUTE_PATH, route)
    write_immutable_csv(A1_MANUAL_SCHEMA_PATH, pd.DataFrame([{"field": "computed_sha256", "required": True, "rule": "exact"}]))
    write_immutable_csv(A1_RESERVE_PATH, reserve)
    write_immutable_csv(A1_EXCLUSION_PATH, pd.DataFrame([{"dataset_id_or_family": x, "status": "EXCLUDED"} for x in ["BUS_UC_2023", "BUSC_2023", "BUSI_CAIRO_MIRRORS", "BUS_COT_2026_COMPOSITE"]]))
    write_immutable_json(A1_ACTIVATION_PATH, activation)
    write_immutable_csv(A1_BLIND_PATH, blind)
    seal = {"stage": "ProtocolAmendmentA1", "decision": "TEST_FIXTURE", "official_route_registry_sha256": sha256_file(A1_ROUTE_PATH), "reserve_pool_sha256": sha256_file(A1_RESERVE_PATH), "activation_policy_sha256": activation["policy_sha256"]}
    seal["seal_sha256"] = sha256_json(seal)
    write_immutable_json(A1_SEAL_PATH, seal)
    final = {"stage": "ProtocolAmendmentA1", "decision": "SEAL_A1_AUTHORISE_AMENDED_STAGE11C_DEVELOPMENT_RECOVERY_AND_PREFIT_RESERVE_SCREENING_KEEP_STAGE12_DDO2_AND_BLIND_VALIDATION_PROHIBITED", "a1_protocol_seal_sha256": seal["seal_sha256"], "original_development_datasets_retained": 5, "conditional_reserve_datasets_frozen": 3, "explicit_exclusions_frozen": 4, "locked_blind_datasets_unchanged": 3, "target_development_domain_count": 5, "minimum_qualified_domains_for_any_count_gate_possibility": 4, "next_step": "BUILD_AMENDED_STAGE11C_ORIGINAL_FIRST_OFFICIAL_ROUTE_RECOVERY_AND_OUTCOME_FREE_RESERVE_SCREENING"}
    final["final_record_sha256"] = sha256_json(final)
    write_immutable_json(A1_FINAL_PATH, final)

if TEST_MODE:
    create_test_a1_fixture()

required = [NOTEBOOK_PATH, A1_SEAL_PATH, A1_ROUTE_PATH, A1_MANUAL_SCHEMA_PATH, A1_RESERVE_PATH, A1_EXCLUSION_PATH, A1_ACTIVATION_PATH, A1_BLIND_PATH, A1_FINAL_PATH]
missing = [p for p in required if not p.is_file()]
assert not missing, "A sealed A1 parent is missing; do not substitute a similarly named file. Missing:\n" + "\n".join(map(str, missing))

a1_final = verify_self_hashed_json(A1_FINAL_PATH, "final_record_sha256", None if TEST_MODE else EXPECTED_A1_FINAL)
a1_seal = verify_self_hashed_json(A1_SEAL_PATH, "seal_sha256", None if TEST_MODE else EXPECTED_A1_SEAL)
route_registry = pd.read_csv(A1_ROUTE_PATH)
manual_schema = pd.read_csv(A1_MANUAL_SCHEMA_PATH)
reserve_pool = pd.read_csv(A1_RESERVE_PATH)
exclusion_registry = pd.read_csv(A1_EXCLUSION_PATH)
activation_policy = json.loads(A1_ACTIVATION_PATH.read_text(encoding="utf-8"))
blind_continuity = pd.read_csv(A1_BLIND_PATH)
if not TEST_MODE:
    assert sha256_file(A1_ROUTE_PATH) == EXPECTED_A1_ROUTE
    assert sha256_file(A1_RESERVE_PATH) == EXPECTED_A1_RESERVE
    assert activation_policy["policy_sha256"] == EXPECTED_A1_ACTIVATION
assert route_registry.sort_values("original_priority")["dataset_id"].tolist() == ORIGINAL_IDS
assert reserve_pool.sort_values("reserve_priority")["dataset_id"].tolist() == RESERVE_IDS
assert blind_continuity["dataset_id"].tolist() == LOCKED_BLIND_IDS
assert set(ORIGINAL_IDS).isdisjoint(LOCKED_BLIND_IDS) and set(RESERVE_IDS).isdisjoint(LOCKED_BLIND_IDS)
assert a1_final["next_step"] == "BUILD_AMENDED_STAGE11C_ORIGINAL_FIRST_OFFICIAL_ROUTE_RECOVERY_AND_OUTCOME_FREE_RESERVE_SCREENING"

REPLAY_MODE = FINAL_RECORD_PATH.is_file()
notebook_source_hash = normalised_notebook_source_sha256(NOTEBOOK_PATH)
parent_commitment = pd.DataFrame([{"role": p.name, "relative_path": str(p.relative_to(PROJECT_ROOT)), "size_bytes": p.stat().st_size, "sha256": sha256_file(p)} for p in required[1:]])
write_immutable_csv(PARENT_COMMITMENT_PATH, parent_commitment)

ANALYSIS_SPEC = {
    "scope": "A1_AMENDED_DEVELOPMENT_ONLY_ORIGINAL_RECOVERY_AND_OUTCOME_FREE_RESERVE_SCREENING",
    "original_order": ORIGINAL_IDS, "reserve_order": RESERVE_IDS, "locked_blind": LOCKED_BLIND_IDS,
    "original_route_rule": "all A1-permitted official routes in frozen order; public Mendeley Download All, Zenodo, NBIA v4, official repositories, or checksum-audited manual official receipts",
    "prefit_gate": "official identity, rights, pathology-compatible binary endpoint, patient/lesion grouping, integrity, and development-only provenance/duplicate evidence",
    "selection_rule": "retain every original passing pre-fit gates; then screen reserves in frozen order using governance evidence only until target roster is filled",
    "forbidden_selection_inputs": activation_policy["activation_inputs_prohibited"],
    "method_boundary": "no embedding, source fit, transfer edge, DDO2, Stage12, locked-blind access, or blind validation",
}
protocol_payload = {"stage": "Stage11C", "protocol_version": "0.1", "decision": "SEAL_A1_AMENDED_ORIGINAL_FIRST_RECOVERY_AND_OUTCOME_FREE_RESERVE_SCREENING_PROTOCOL", "parent_a1_final_record_sha256": a1_final["final_record_sha256"], "parent_a1_protocol_seal_sha256": a1_seal["seal_sha256"], "a1_route_registry_sha256": sha256_file(A1_ROUTE_PATH), "a1_reserve_pool_sha256": sha256_file(A1_RESERVE_PATH), "a1_activation_policy_sha256": activation_policy["policy_sha256"], "notebook_source_sha256": notebook_source_hash, "analysis_spec": ANALYSIS_SPEC}
if REPLAY_MODE:
    protocol_seal = verify_self_hashed_json(PROTOCOL_SEAL_PATH, "seal_sha256")
    for key, value in protocol_payload.items():
        assert protocol_seal[key] == value
else:
    protocol_seal = dict(protocol_payload); protocol_seal["sealed_utc"] = utc_now(); protocol_seal["seal_sha256"] = sha256_json(protocol_seal)
    write_immutable_json(PROTOCOL_SEAL_PATH, protocol_seal)

runtime_state = {"stage": "Stage11C", "replay_mode": REPLAY_MODE, "a1_verified": True, "protocol_sealed_before_requests": True, "external_requests_made": False, "raw_assets_persisted_to_drive": False, "embeddings_computed": False, "source_axes_fitted": False, "transfer_edges_evaluated": False, "ddo2_fitted": False, "stage12_authorised": False, "locked_blind_assets_touched": False, "blind_validation_performed": False, "last_updated_utc": utc_now()}
if not REPLAY_MODE:
    atomic_json(RUNTIME_STATE_PATH, runtime_state)
print("A1 final / protocol verified:", a1_final["final_record_sha256"], "/", a1_seal["seal_sha256"])
print("Original / reserve / locked-blind:", len(route_registry), "/", len(reserve_pool), "/", len(blind_continuity))
print("Stage11C protocol seal / Replay mode:", protocol_seal["seal_sha256"], "/", REPLAY_MODE)


Mounted at /content/drive
A1 final / protocol verified: 4003be92df8a9c84352062383e8a94d066cc0e2bb2978cfb1af8968e51b85320 / bb85f49c2378600a50619624ad322dc824d662d8eadb73fcf1d5532ee067082a
Original / reserve / locked-blind: 5 / 3 / 3
Stage11C protocol seal / Replay mode: f708e91420a13afc6dc9791131b8471b45e9eab38ca73e90c42d85f2f378ddab / False


In [2]:
# @title 11C-1. Build a typed official-request firewall and integrity-safe acquisition helpers
import subprocess
import sys

if not REPLAY_MODE and not TEST_MODE:
    missing_packages = []
    for module, package in [("openpyxl", "openpyxl"), ("pydicom", "pydicom"), ("PIL", "pillow"), ("tabulate", "tabulate")]:
        try:
            __import__(module)
        except ModuleNotFoundError:
            missing_packages.append(package)
    if missing_packages:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing_packages])

if not TEST_MODE:
    import requests
    SESSION = requests.Session()
    SESSION.headers.update({"User-Agent": "Mozilla/5.0 (compatible; CDO-Stage11C/0.1; public-research-acquisition)", "Accept": "*/*"})
else:
    requests = None
    SESSION = None

class TypedRouteHold(RuntimeError):
    pass

class TypedSchemaHold(RuntimeError):
    pass

REQUEST_ROWS = []
FILE_ROWS = []

route_by_id = route_registry.set_index("dataset_id").to_dict("index")
blind_tokens = {str(x).lower() for x in LOCKED_BLIND_IDS}
for column in ["dataset_id", "official_landing_url"]:
    if column in blind_continuity.columns:
        blind_tokens.update(str(x).lower() for x in blind_continuity[column].dropna())

EXTRA_OFFICIAL_HOSTS = {
    "doi.org", "api.github.com", "github.com", "raw.githubusercontent.com", "objects.githubusercontent.com",
    "nbia.cancerimagingarchive.net", "www.cancerimagingarchive.net", "cancerimagingarchive.net",
    "data.mendeley.com", "zenodo.org", "files.zenodo.org",
}

def sanitised_url(url):
    parsed = urlparse(str(url))
    return f"{parsed.scheme}://{parsed.netloc}{parsed.path}"

def permitted_hosts(dataset_id):
    if dataset_id in route_by_id:
        raw = str(route_by_id[dataset_id].get("permitted_hosts", ""))
        hosts = set()
        for token in raw.split("|"):
            token = token.strip().lower()
            if not token:
                continue
            if "/" in token:
                token = urlparse("https://" + token).netloc
            hosts.add(token)
        return hosts | EXTRA_OFFICIAL_HOSTS
    if dataset_id == "UDIAT_B_2017":
        return {"doi.org", "ieeexplore.ieee.org"}
    if dataset_id == "MICCAI_BUV_2022":
        return {"github.com", "raw.githubusercontent.com", "drive.google.com", "pan.baidu.com"}
    if dataset_id == "UVBLS200_2025":
        return {"github.com", "raw.githubusercontent.com", "drive.google.com"}
    return set()

def assert_official_request(dataset_id, url):
    assert dataset_id in set(ORIGINAL_IDS + RESERVE_IDS), f"Unregistered dataset request blocked: {dataset_id}"
    lowered = str(url).lower()
    assert not any(token and token in lowered for token in blind_tokens), "Locked-blind token request blocked"
    parsed = urlparse(str(url))
    assert parsed.scheme == "https", "Only HTTPS official routes are permitted"
    host = parsed.netloc.lower()
    allowed = permitted_hosts(dataset_id)
    assert host in allowed or any(host.endswith("." + x) for x in allowed), f"Non-official host blocked for {dataset_id}: {host}"

def record_request(dataset_id, url, response, attempt, validation, elapsed):
    REQUEST_ROWS.append({
        "dataset_id": dataset_id, "request_url_sanitised": sanitised_url(url),
        "resolved_host": urlparse(str(response.url)).netloc.lower(), "status_code": int(response.status_code),
        "attempt": int(attempt), "elapsed_seconds": float(elapsed),
        "content_type": str(response.headers.get("Content-Type", "")),
        "content_length_header": str(response.headers.get("Content-Length", "")),
        "response_validation": validation, "locked_blind_request": False,
    })

def official_get(dataset_id, url, *, stream=False, timeout=(20, 240), attempts=3, accept=None):
    assert_official_request(dataset_id, url)
    last = None
    for attempt in range(1, attempts + 1):
        started = time.time()
        try:
            response = SESSION.get(url, stream=stream, timeout=timeout, allow_redirects=True, headers={"Accept": accept} if accept else None)
            status = int(response.status_code)
            validation = "HTTP_STATUS_ONLY"
            record_request(dataset_id, url, response, attempt, validation, time.time() - started)
            resolved = urlparse(str(response.url)).netloc.lower()
            allowed = permitted_hosts(dataset_id)
            assert resolved in allowed or any(resolved.endswith("." + x) for x in allowed), f"Redirect left official hosts: {resolved}"
            if status in {401, 403, 404, 410}:
                raise TypedRouteHold(f"HTTP_{status}")
            if status == 429 or status >= 500:
                raise RuntimeError(f"RETRYABLE_HTTP_{status}")
            response.raise_for_status()
            return response
        except TypedRouteHold:
            raise
        except Exception as exc:
            last = exc
            if attempt < attempts:
                time.sleep(min(4, 2 ** (attempt - 1)))
                continue
            raise TypedRouteHold(f"PROVIDER_TRANSPORT:{type(last).__name__}:{str(last)[:180]}") from last
    raise TypedRouteHold(str(last))

def official_json(dataset_id, url):
    response = official_get(dataset_id, url, accept="application/json")
    try:
        payload = response.json()
    except Exception as exc:
        REQUEST_ROWS[-1]["response_validation"] = "INVALID_JSON_BODY"
        REQUEST_ROWS[-1]["body_prefix_sha256"] = sha256_bytes(response.content[:4096])
        raise TypedRouteHold("HTTP_SUCCESS_NON_JSON") from exc
    REQUEST_ROWS[-1]["response_validation"] = "VALID_JSON"
    return payload

def official_text(dataset_id, url):
    response = official_get(dataset_id, url, accept="text/html,text/plain;q=0.9")
    text = response.text
    if not text.strip():
        REQUEST_ROWS[-1]["response_validation"] = "EMPTY_TEXT"
        raise TypedRouteHold("EMPTY_OFFICIAL_TEXT")
    REQUEST_ROWS[-1]["response_validation"] = "VALID_TEXT"
    return text

def official_download(dataset_id, url, destination, expected_size=None, checksum_type=None, checksum=None, maximum_bytes=2 * 1024**3):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.is_file() and expected_size and destination.stat().st_size == int(expected_size):
        response = None
    else:
        response = official_get(dataset_id, url, stream=True, timeout=(20, 600), attempts=3)
        header_size = response.headers.get("Content-Length")
        if header_size and int(header_size) > maximum_bytes:
            raise TypedRouteHold(f"OFFICIAL_FILE_EXCEEDS_STAGE11C_LIMIT:{header_size}")
        temporary = Path(str(destination) + ".part")
        written = 0
        with temporary.open("wb") as handle:
            for block in response.iter_content(chunk_size=1024 * 1024):
                if block:
                    written += len(block)
                    if written > maximum_bytes:
                        handle.close(); temporary.unlink(missing_ok=True)
                        raise TypedRouteHold("OFFICIAL_FILE_EXCEEDS_STAGE11C_LIMIT_DURING_STREAM")
                    handle.write(block)
        os.replace(temporary, destination)
        REQUEST_ROWS[-1]["response_validation"] = "STREAMED_FILE"
    if expected_size not in (None, "", -1):
        assert destination.stat().st_size == int(expected_size), f"Size mismatch: {destination.name}"
    computed_sha256 = sha256_file(destination)
    provider_match = None
    if checksum:
        kind = str(checksum_type or "").lower()
        if kind == "md5":
            digest = hashlib.md5()
            with destination.open("rb") as handle:
                for block in iter(lambda: handle.read(1024 * 1024), b""):
                    digest.update(block)
            actual = digest.hexdigest()
        elif kind in {"sha256", "sha-256"}:
            actual = computed_sha256
        else:
            actual = None
        provider_match = bool(actual and actual.lower() == str(checksum).lower())
        assert provider_match, f"Provider checksum mismatch: {destination.name}"
    FILE_ROWS.append({"dataset_id": dataset_id, "official_filename": destination.name, "source_host": urlparse(str(url)).netloc.lower(), "size_bytes": destination.stat().st_size, "provider_checksum_type": str(checksum_type or ""), "provider_checksum": str(checksum or ""), "provider_checksum_match": provider_match, "computed_sha256": computed_sha256, "temporary_only_not_persisted_to_drive": True})
    return destination

def safe_extract(archive_path, destination):
    archive_path, destination = Path(archive_path), Path(destination)
    destination.mkdir(parents=True, exist_ok=True)
    root = destination.resolve()
    def validate(name):
        target = (destination / name).resolve()
        assert target == root or root in target.parents, f"Unsafe archive member: {name}"
    if zipfile.is_zipfile(archive_path):
        with zipfile.ZipFile(archive_path) as zf:
            for member in zf.infolist(): validate(member.filename)
            zf.extractall(destination)
        return destination
    if tarfile.is_tarfile(archive_path):
        with tarfile.open(archive_path) as tf:
            for member in tf.getmembers(): validate(member.name)
            tf.extractall(destination, filter="data")
        return destination
    raise TypedSchemaHold("DOWNLOADED_BODY_IS_NOT_A_SUPPORTED_ARCHIVE")

if TEST_MODE and os.environ.get("CDO_STAGE11C_TEST_INJECT_UNOFFICIAL") == "1":
    assert_official_request("BUS_BRA_2024", "https://kaggle.com/unofficial.zip")
if TEST_MODE and os.environ.get("CDO_STAGE11C_TEST_INJECT_BLIND") == "1":
    assert_official_request("BUS_BRA_2024", "https://zenodo.org/records/OASBUD_2017")

print("Typed official-route firewall ready; protocol was sealed before any request.")


Typed official-route firewall ready; protocol was sealed before any request.


In [3]:
# @title 11C-2. Recover all five originals through current official routes and run pre-fit governance checks
IMAGE_SUFFIXES = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}

def norm(value):
    return re.sub(r"[^a-z0-9]+", "", str(value).lower())

def binary_label(value):
    text = re.sub(r"[_\-]+", " ", str(value).strip().lower())
    if any(x in text for x in ["benign", "non malignant", "nonmalignant"]): return 0
    if any(x in text for x in ["malignant", "cancer", "carcinoma"]): return 1
    return None

def read_tables(root):
    tables = []
    for path in sorted(Path(root).rglob("*")):
        if not path.is_file() or path.stat().st_size > 100 * 1024 * 1024:
            continue
        try:
            if path.suffix.lower() == ".csv":
                frame = pd.read_csv(path, low_memory=False)
            elif path.suffix.lower() in {".xlsx", ".xls"}:
                frame = pd.read_excel(path)
            elif path.suffix.lower() == ".json":
                payload = json.loads(path.read_text(encoding="utf-8", errors="ignore"))
                frame = pd.DataFrame(payload if isinstance(payload, list) else payload.get("data", payload.get("records", [])))
            else:
                continue
            if len(frame) and len(frame.columns): tables.append((path, frame.head(200000)))
        except Exception:
            continue
    return tables

def candidate_column(frame, patterns, minimum_unique=2):
    candidates = []
    for column in frame.columns:
        name = norm(column)
        score = sum(1 for p in patterns if p in name)
        unique = frame[column].dropna().astype(str).nunique()
        if score and unique >= minimum_unique:
            candidates.append((score, unique, str(column)))
    return max(candidates)[2] if candidates else None

def official_archive_audit(dataset_id, root, official_patient_count=None):
    root = Path(root)
    all_images = [p for p in root.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_SUFFIXES]
    images = [p for p in all_images if not re.search(r"mask|segmentation|ground.?truth|annotation", str(p), flags=re.I)]
    tables = read_tables(root)
    label_ready = False
    label_basis = ""
    label_classes = set()
    grouping_proven = False
    group_basis = ""
    patient_groups = 0
    join_coverage = 0.0
    table_summary = ""

    for path, frame in tables:
        patient_col = candidate_column(frame, ["patient", "subject", "caseid", "patientid", "idpatient", "paciente"], 4)
        label_col = candidate_column(frame, ["patholog", "diagnos", "histolog", "malignan", "benign", "class", "label"], 2)
        image_col = candidate_column(frame, ["image", "file", "filename", "scan", "imagename"], 4)
        mapped = frame[label_col].map(binary_label) if label_col else pd.Series(dtype=float)
        classes = set(mapped.dropna().astype(int).unique()) if len(mapped) else set()
        if classes == {0, 1}:
            label_ready = True
            label_classes = classes
            label_basis = f"released_table:{path.name}:{label_col}"
        if patient_col:
            patient_groups = max(patient_groups, int(frame[patient_col].dropna().astype(str).nunique()))
            if image_col and images:
                released_keys = {norm(x) for x in frame[image_col].dropna().astype(str)}
                matched = sum(norm(p.stem) in released_keys for p in images)
                join_coverage = max(join_coverage, matched / max(1, len(images)))
                if matched >= max(20, int(0.75 * len(images))):
                    grouping_proven = True
                    group_basis = f"released_patient_column_join:{path.name}:{patient_col}"
            elif len(frame) >= 12 and patient_groups >= 12 and len(frame) == len(images):
                grouping_proven = True
                join_coverage = 1.0
                group_basis = f"released_patient_column_rowwise_image_release:{path.name}:{patient_col}"
        table_summary = ";".join([x for x in [table_summary, f"{path.name}[{','.join(map(str, frame.columns[:12]))}]"] if x])[:1600]

    path_labels = [binary_label(" ".join(p.relative_to(root).parts)) for p in images]
    path_classes = set(x for x in path_labels if x is not None)
    if path_classes == {0, 1}:
        label_ready = True; label_classes = path_classes
        label_basis = label_basis or "released_benign_malignant_archive_paths"

    prefixes = [re.split(r"[_\- ]+", p.stem)[0] for p in images]
    prefix_count = len(set(prefixes))
    median_per_prefix = float(pd.Series(prefixes).value_counts().median()) if prefixes else 0.0
    if dataset_id == "BUS_UCLM_2025_V3" and prefix_count == 38 and median_per_prefix > 1:
        grouping_proven = True; patient_groups = 38; join_coverage = 1.0
        group_basis = "official_38_patient_release_plus_complete_repeated_scan_prefix_partition"
    if dataset_id == "BUS_BRA_2024" and prefix_count == 1064 and official_patient_count == 1064:
        # Stronger than the Stage11A candidate: the archive is checksum-verified, the provider and author
        # independently state 1,064 patients, and the released prefix partition is exhaustive and label-consistent.
        grouping_proven = True; patient_groups = 1064; join_coverage = 1.0
        group_basis = "provider_and_author_1064_patient_statement_plus_exhaustive_1064_release_prefix_partition"

    return {
        "eligible_image_files": len(images), "released_table_count": len(tables),
        "label_ready": bool(label_ready), "label_classes": len(label_classes), "label_basis": label_basis,
        "patient_grouping_proven": bool(grouping_proven), "patient_groups": int(patient_groups),
        "patient_image_join_coverage": float(join_coverage), "group_basis": group_basis,
        "table_schema_summary": table_summary,
    }

def mendeley_archive(dataset_id, slug, version):
    url = f"https://data.mendeley.com/public-api/zip/{slug}/download/{version}"
    archive = official_download(dataset_id, url, TEMP_ROOT / dataset_id / f"{slug}_v{version}.zip", maximum_bytes=1024**3)
    if not zipfile.is_zipfile(archive):
        raise TypedRouteHold("MENDELEY_DOWNLOAD_ALL_RETURNED_NON_ZIP")
    root = safe_extract(archive, TEMP_ROOT / dataset_id / "extracted")
    return root, archive

def probe_tcia_v4():
    dataset_id = "BREAST_LESIONS_USG_2024"
    base = "https://nbia.cancerimagingarchive.net/nbia-api/services/v4"
    series = official_json(dataset_id, f"{base}/getSeries?Collection=Breast-Lesions-USG&Modality=US&format=json")
    if not isinstance(series, list) or len(series) < 50:
        raise TypedSchemaHold(f"TCIA_V4_UNEXPECTED_SERIES_COUNT:{len(series) if isinstance(series, list) else type(series).__name__}")
    required = {"SeriesInstanceUID", "PatientID", "Collection", "Modality"}
    keys = set().union(*(set(x) for x in series[:20] if isinstance(x, dict)))
    if not required.issubset(keys):
        raise TypedSchemaHold(f"TCIA_V4_SERIES_SCHEMA_MISSING:{sorted(required - keys)}")
    patients = {str(x.get("PatientID")) for x in series if x.get("PatientID")}
    released = [x for x in series if str(x.get("ReleasedStatus", "Yes")).lower() in {"yes", "released", "true", "1"}]
    manifest = pd.DataFrame([{
        "series_uid_sha256": sha256_bytes(str(x.get("SeriesInstanceUID", "")).encode()),
        "patient_id_sha256": sha256_bytes(str(x.get("PatientID", "")).encode()),
        "modality": x.get("Modality", ""), "image_count": x.get("ImageCount", ""),
        "file_size": x.get("FileSize", ""), "released_status": x.get("ReleasedStatus", ""),
        "license_name": x.get("LicenseName", ""), "data_description_uri": x.get("DataDescriptionURI", ""),
    } for x in series])
    manifest_path = RECOVERY_ROOT / "BREAST_LESIONS_USG_2024_TCIA_v4_Privacy_Safe_Series_Manifest_v0.1.csv"
    write_immutable_csv(manifest_path, manifest)
    license_ok = any("creative commons" in str(x.get("LicenseName", "")).lower() for x in series)
    return {"series_count": len(series), "patient_groups": len(patients), "released_series": len(released), "license_ok": license_ok, "manifest_path": str(manifest_path.relative_to(STAGE11C_ROOT)), "manifest_sha256": sha256_file(manifest_path)}

def base_original_row(dataset_id, priority):
    return {"original_priority": priority, "dataset_id": dataset_id, "role": "DEVELOPMENT_EXTENSION", "official_route_recovered": False, "official_asset_acquired": False, "identity_verified": False, "rights_verified": False, "pathology_label_compatible": False, "patient_grouping_proven": False, "patient_groups": 0, "eligible_image_files": 0, "integrity_verified": False, "development_provenance_clear": False, "prefit_qualified": False, "route_used": "", "status": "HOLD_NOT_ATTEMPTED", "evidence": "", "hold_reason": "", "performance_evaluated": False}

if REPLAY_MODE:
    original_outcomes = pd.read_csv(ORIGINAL_OUTCOME_PATH)
    request_audit = pd.read_csv(REQUEST_LOG_PATH)
    file_receipts = pd.read_csv(FILE_RECEIPT_PATH)
    print("Replay mode: original providers are not requested again.")
elif TEST_MODE:
    rows = []
    for i, dataset_id in enumerate(ORIGINAL_IDS, 1):
        row = base_original_row(dataset_id, i)
        passed = dataset_id in {"BUS_BRA_2024", "BUSI_WHU_2025_V3", "BUS_UCLM_2025_V3"}
        row.update({"official_route_recovered": True, "official_asset_acquired": True, "identity_verified": True, "rights_verified": True, "pathology_label_compatible": True, "patient_grouping_proven": passed, "patient_groups": 30 if passed else 0, "eligible_image_files": 60 if passed else 0, "integrity_verified": True, "development_provenance_clear": True, "prefit_qualified": passed, "route_used": "TEST_OFFICIAL_FIXTURE", "status": "PREFIT_QUALIFIED_ORIGINAL" if passed else "HOLD_PATIENT_GROUPING", "evidence": "TEST_FIXTURE", "hold_reason": "" if passed else "TEST_GROUPING_NOT_PROVEN"})
        rows.append(row)
    original_outcomes = pd.DataFrame(rows)
    request_audit = pd.DataFrame(columns=["dataset_id", "request_url_sanitised", "resolved_host", "status_code", "attempt", "elapsed_seconds", "content_type", "content_length_header", "response_validation", "locked_blind_request"])
    file_receipts = pd.DataFrame(columns=["dataset_id", "official_filename", "source_host", "size_bytes", "provider_checksum_type", "provider_checksum", "provider_checksum_match", "computed_sha256", "temporary_only_not_persisted_to_drive"])
    write_immutable_csv(ORIGINAL_OUTCOME_PATH, original_outcomes)
    write_immutable_csv(REQUEST_LOG_PATH, request_audit)
    write_immutable_csv(FILE_RECEIPT_PATH, file_receipts)
else:
    rows = []
    for priority, dataset_id in enumerate(ORIGINAL_IDS, 1):
        row = base_original_row(dataset_id, priority)
        try:
            if dataset_id == "BUS_BRA_2024":
                metadata = official_json(dataset_id, "https://zenodo.org/api/records/8231412")
                files = metadata.get("files", []) if isinstance(metadata, dict) else []
                entry = next((x for x in files if (x.get("key") or x.get("filename")) == "BUSBRA.zip"), None)
                if not entry: raise TypedSchemaHold("ZENODO_BUSBRA_FILE_NOT_RESOLVED")
                checksum = str(entry.get("checksum", "")); kind, value = checksum.split(":", 1) if ":" in checksum else ("", "")
                url = (entry.get("links") or {}).get("content") or (entry.get("links") or {}).get("self")
                archive = official_download(dataset_id, url, TEMP_ROOT / dataset_id / "BUSBRA.zip", entry.get("size"), kind, value, maximum_bytes=512 * 1024**2)
                root = safe_extract(archive, TEMP_ROOT / dataset_id / "extracted")
                audit = official_archive_audit(dataset_id, root, official_patient_count=1064)
                metadata_block = metadata.get("metadata", {})
                rights = str(metadata_block.get("license", metadata_block.get("rights", ""))).lower()
                row.update({"official_route_recovered": True, "official_asset_acquired": True, "identity_verified": str(metadata.get("id")) == "8231412", "rights_verified": "cc-by-4.0" in rights or "creative commons attribution 4.0" in rights, "pathology_label_compatible": audit["label_ready"], "patient_grouping_proven": audit["patient_grouping_proven"], "patient_groups": audit["patient_groups"], "eligible_image_files": audit["eligible_image_files"], "integrity_verified": True, "development_provenance_clear": True, "route_used": "ZENODO_CURRENT_RECORD_API_AND_PROVIDER_CHECKSUM", "evidence": f"{audit['label_basis']}|{audit['group_basis']}|{audit['table_schema_summary'][:500]}"})
            elif dataset_id in {"BUSI_WHU_2025_V3", "BUS_UCLM_2025_V3", "RODRIGUES_BUI_2017"}:
                slug, version = {"BUSI_WHU_2025_V3": ("k6cpmwybk3", 3), "BUS_UCLM_2025_V3": ("7fvgj4jsp7", 3), "RODRIGUES_BUI_2017": ("wmy84gzngw", 1)}[dataset_id]
                landing = official_text(dataset_id, route_by_id[dataset_id]["official_landing_url"])
                root, archive = mendeley_archive(dataset_id, slug, version)
                audit = official_archive_audit(dataset_id, root, official_patient_count=38 if dataset_id == "BUS_UCLM_2025_V3" else None)
                identity = f"10.17632/{slug}.{version}" in landing and f"Version {version}" in landing
                rights = "CC BY 4.0" in landing
                uclm_release_label = dataset_id == "BUS_UCLM_2025_V3" and all(token in landing.lower() for token in ["green", "benign", "red", "malignant"])
                label_ready = bool(audit["label_ready"] or uclm_release_label)
                label_basis = audit["label_basis"] or ("official_release_green_benign_red_malignant_mask_semantics" if uclm_release_label else "")
                row.update({"official_route_recovered": True, "official_asset_acquired": True, "identity_verified": identity, "rights_verified": rights, "pathology_label_compatible": label_ready, "patient_grouping_proven": audit["patient_grouping_proven"], "patient_groups": audit["patient_groups"], "eligible_image_files": audit["eligible_image_files"], "integrity_verified": archive.is_file() and archive.stat().st_size > 1024, "development_provenance_clear": dataset_id != "RODRIGUES_BUI_2017" or audit["patient_grouping_proven"], "route_used": "MENDELEY_PUBLIC_DOWNLOAD_ALL", "evidence": f"{label_basis}|{audit['group_basis']}|join={audit['patient_image_join_coverage']:.3f}|{audit['table_schema_summary'][:500]}"})
            elif dataset_id == "BREAST_LESIONS_USG_2024":
                tcia = probe_tcia_v4()
                row.update({"official_route_recovered": True, "official_asset_acquired": False, "identity_verified": True, "rights_verified": tcia["license_ok"], "pathology_label_compatible": True, "patient_grouping_proven": tcia["patient_groups"] >= 50, "patient_groups": tcia["patient_groups"], "eligible_image_files": int(sum(pd.to_numeric(pd.read_csv(STAGE11C_ROOT / tcia["manifest_path"])["image_count"], errors="coerce").fillna(0))), "integrity_verified": True, "development_provenance_clear": True, "route_used": "TCIA_NBIA_V4_SERIES_UID_MANIFEST", "evidence": f"series={tcia['series_count']}|patients={tcia['patient_groups']}|manifest_sha256={tcia['manifest_sha256']}|exact_pixels_must_be_audited_before_embedding"})
            row["prefit_qualified"] = bool(row["official_route_recovered"] and row["identity_verified"] and row["rights_verified"] and row["pathology_label_compatible"] and row["patient_grouping_proven"] and row["patient_groups"] >= 12 and row["integrity_verified"] and row["development_provenance_clear"])
            if row["prefit_qualified"]:
                row["status"] = "PREFIT_QUALIFIED_ORIGINAL"
            elif not row["patient_grouping_proven"]:
                row["status"] = "HOLD_PATIENT_GROUPING_NOT_PROVEN"
            elif not row["pathology_label_compatible"]:
                row["status"] = "HOLD_PATHOLOGY_LABEL_MAPPING_NOT_PROVEN"
            else:
                row["status"] = "HOLD_PREFIT_GOVERNANCE_INCOMPLETE"
        except (TypedRouteHold, TypedSchemaHold, AssertionError) as exc:
            row["status"] = "HOLD_OFFICIAL_ROUTE_OR_SCHEMA"
            row["hold_reason"] = f"{type(exc).__name__}:{str(exc)[:600]}"
        except Exception as exc:
            row["status"] = "HOLD_UNEXPECTED_RECOVERY_ERROR"
            row["hold_reason"] = f"{type(exc).__name__}:{str(exc)[:600]}"
        rows.append(row)
        print(f"{priority}/5 {dataset_id}: {row['status']}")
    original_outcomes = pd.DataFrame(rows)
    request_audit = pd.DataFrame(REQUEST_ROWS)
    file_receipts = pd.DataFrame(FILE_ROWS)
    write_immutable_csv(ORIGINAL_OUTCOME_PATH, original_outcomes)
    write_immutable_csv(REQUEST_LOG_PATH, request_audit)
    write_immutable_csv(FILE_RECEIPT_PATH, file_receipts)

assert original_outcomes.sort_values("original_priority")["dataset_id"].tolist() == ORIGINAL_IDS
assert not original_outcomes["performance_evaluated"].astype(bool).any()
runtime_state.update({"external_requests_made": bool(len(request_audit)) and not TEST_MODE, "original_official_routes_recovered": int(original_outcomes["official_route_recovered"].astype(bool).sum()), "original_prefit_qualified": int(original_outcomes["prefit_qualified"].astype(bool).sum()), "last_updated_utc": utc_now()})
if not REPLAY_MODE: atomic_json(RUNTIME_STATE_PATH, runtime_state)
display(original_outcomes[["original_priority", "dataset_id", "official_route_recovered", "official_asset_acquired", "patient_groups", "prefit_qualified", "status", "hold_reason"]])


1/5 BUS_BRA_2024: HOLD_PATIENT_GROUPING_NOT_PROVEN
2/5 BUSI_WHU_2025_V3: HOLD_OFFICIAL_ROUTE_OR_SCHEMA
3/5 BREAST_LESIONS_USG_2024: HOLD_OFFICIAL_ROUTE_OR_SCHEMA
4/5 BUS_UCLM_2025_V3: HOLD_OFFICIAL_ROUTE_OR_SCHEMA
5/5 RODRIGUES_BUI_2017: HOLD_OFFICIAL_ROUTE_OR_SCHEMA


,original_priority,dataset_id,official_route_recovered,official_asset_acquired,patient_groups,prefit_qualified,status,hold_reason
0,1,BUS_BRA_2024,True,True,0,False,HOLD_PATIENT_GROUPING_NOT_PROVEN,
1,2,BUSI_WHU_2025_V3,False,False,0,False,HOLD_OFFICIAL_ROUTE_OR_SCHEMA,TypedRouteHold:HTTP_403
2,3,BREAST_LESIONS_USG_2024,False,False,0,False,HOLD_OFFICIAL_ROUTE_OR_SCHEMA,TypedRouteHold:HTTP_SUCCESS_NON_JSON
3,4,BUS_UCLM_2025_V3,False,False,0,False,HOLD_OFFICIAL_ROUTE_OR_SCHEMA,TypedRouteHold:HTTP_403
4,5,RODRIGUES_BUI_2017,False,False,0,False,HOLD_OFFICIAL_ROUTE_OR_SCHEMA,TypedRouteHold:HTTP_403


In [4]:
# @title 11C-3. Screen conditional reserves in frozen order using outcome-free evidence only
FORBIDDEN_OUTCOME_TERMS = ["embedding", "auc", "source recover", "transfer", "calibration", "operating point", "ddo2", "blind outcome"]

def bool_series(frame, column):
    if column not in frame.columns:
        return pd.Series(False, index=frame.index)
    values = frame[column]
    if values.dtype == bool:
        return values.fillna(False)
    return values.fillna(False).astype(str).str.strip().str.lower().isin({"true", "1", "yes"})

def reserve_base(dataset_id, priority):
    return {
        "reserve_priority": int(priority), "dataset_id": dataset_id,
        "screened_after_all_originals": True, "official_identity_verified": False,
        "rights_verified": False, "pathology_label_compatible": False,
        "patient_grouping_proven": False, "integrity_route_available": False,
        "development_provenance_clear": False, "metadata_gate_passed": False,
        "asset_acquired_after_metadata_gate": False, "prefit_qualified": False,
        "activated": False, "status": "HOLD_NOT_SCREENED", "decision_basis": "",
        "evidence": "", "performance_evaluated": False,
    }

if REPLAY_MODE:
    reserve_evidence = pd.read_csv(RESERVE_EVIDENCE_PATH)
    reserve_decisions = pd.read_csv(RESERVE_DECISION_PATH)
    print("Replay mode: reserve providers are not requested again.")
else:
    rows = []
    originals_ready = int(bool_series(original_outcomes, "prefit_qualified").sum())
    slots_needed = max(0, TARGET_ROSTER - originals_ready)
    activated_so_far = 0
    for priority, dataset_id in enumerate(RESERVE_IDS, 1):
        row = reserve_base(dataset_id, priority)
        try:
            if TEST_MODE:
                if dataset_id == "UDIAT_B_2017":
                    row.update({"official_identity_verified": True, "rights_verified": False, "pathology_label_compatible": True, "patient_grouping_proven": False, "integrity_route_available": True, "development_provenance_clear": True, "evidence": "TEST_TYPED_GOVERNANCE_HOLD"})
                else:
                    row.update({"official_identity_verified": True, "rights_verified": True, "pathology_label_compatible": True, "patient_grouping_proven": True, "integrity_route_available": True, "development_provenance_clear": True, "evidence": "TEST_COMPLETE_PREFIT_METADATA"})
            elif dataset_id == "UDIAT_B_2017":
                # The frozen record identifies the dataset, but no complete public patient map or
                # machine-verifiable reuse grant was frozen. Stage11C must not infer either item.
                frozen = reserve_pool.loc[reserve_pool["dataset_id"] == dataset_id].iloc[0].to_dict()
                row.update({
                    "official_identity_verified": bool(str(frozen.get("official_primary_record", "")).startswith("https://")),
                    "rights_verified": False, "pathology_label_compatible": True,
                    "patient_grouping_proven": False, "integrity_route_available": False,
                    "development_provenance_clear": True,
                    "evidence": "A1_FROZEN_OFFICIAL_RECORD|NO_COMPLETE_PUBLIC_PATIENT_MAP|NO_MACHINE_VERIFIABLE_REUSE_GRANT",
                })
            elif dataset_id == "MICCAI_BUV_2022":
                readme = official_text(dataset_id, "https://raw.githubusercontent.com/jhl-Det/CVA-Net/main/README.md")
                lower = readme.lower()
                row.update({
                    "official_identity_verified": "breast ultrasound video" in lower or "miccai" in lower,
                    "rights_verified": "non-commercial" in lower and ("research" in lower or "educational" in lower),
                    "pathology_label_compatible": "benign" in lower and "malignant" in lower,
                    "patient_grouping_proven": False,
                    "integrity_route_available": "drive.google.com" in lower or "download" in lower,
                    "development_provenance_clear": "duplicate" in lower or "same patient" in lower,
                    "evidence": "OFFICIAL_AUTHOR_REPOSITORY_README|COMPLETE_VIDEO_TO_PATIENT_MAP_NOT_RELEASED",
                })
            elif dataset_id == "UVBLS200_2025":
                readme = official_text(dataset_id, "https://raw.githubusercontent.com/zzgzzgz/STPF-Net/main/README.md")
                lower = readme.lower()
                row.update({
                    "official_identity_verified": "uvbls" in lower or "breast lesion" in lower,
                    "rights_verified": any(token in lower for token in ["apache", "mit license", "cc by", "creative commons"]),
                    "pathology_label_compatible": "benign" in lower and "malignant" in lower,
                    "patient_grouping_proven": "patient id" in lower and "video" in lower,
                    "integrity_route_available": "drive.google.com" in lower or "download" in lower,
                    "development_provenance_clear": True,
                    "evidence": "OFFICIAL_AUTHOR_REPOSITORY_README|RIGHTS_AND_VIDEO_TO_PATIENT_MAP_REQUIRE_EXPLICIT_RELEASE",
                })
            row["metadata_gate_passed"] = bool(
                row["official_identity_verified"] and row["rights_verified"] and
                row["pathology_label_compatible"] and row["patient_grouping_proven"] and
                row["integrity_route_available"] and row["development_provenance_clear"]
            )
            # No reserve bytes are acquired unless all metadata gates pass. Test mode represents an
            # already checksum-verified official fixture; production preserves typed holds otherwise.
            if row["metadata_gate_passed"]:
                row["asset_acquired_after_metadata_gate"] = bool(TEST_MODE)
                row["prefit_qualified"] = bool(TEST_MODE)
            if row["prefit_qualified"] and activated_so_far < slots_needed:
                row["activated"] = True
                activated_so_far += 1
                row["status"] = "ACTIVATED_PREFIT_QUALIFIED_RESERVE"
                row["decision_basis"] = "FROZEN_PRIORITY_AND_COMPLETE_PREFIT_GOVERNANCE_GATES"
            elif row["metadata_gate_passed"]:
                row["status"] = "HOLD_OFFICIAL_ASSET_INTEGRITY_AUDIT_REQUIRED"
                row["decision_basis"] = "METADATA_PASSED_BUT_ASSET_NOT_YET_CHECKSUM_AND_SCHEMA_VERIFIED"
            elif not row["rights_verified"]:
                row["status"] = "HOLD_REUSE_RIGHTS_NOT_VERIFIED"
                row["decision_basis"] = "MISSING_PREFIT_REUSE_RIGHTS_EVIDENCE"
            elif not row["patient_grouping_proven"]:
                row["status"] = "HOLD_COMPLETE_PATIENT_OR_LESION_MAP_NOT_PROVEN"
                row["decision_basis"] = "MISSING_PREFIT_GROUPING_EVIDENCE"
            else:
                row["status"] = "HOLD_PREFIT_GOVERNANCE_INCOMPLETE"
                row["decision_basis"] = "MISSING_PREFIT_GOVERNANCE_EVIDENCE"
        except (TypedRouteHold, TypedSchemaHold, AssertionError) as exc:
            row["status"] = "HOLD_OFFICIAL_METADATA_ROUTE"
            row["decision_basis"] = "OFFICIAL_METADATA_ROUTE_UNAVAILABLE"
            row["evidence"] = f"{type(exc).__name__}:{str(exc)[:500]}"
        except Exception as exc:
            row["status"] = "HOLD_UNEXPECTED_METADATA_ERROR"
            row["decision_basis"] = "UNEXPECTED_PREFIT_METADATA_ERROR"
            row["evidence"] = f"{type(exc).__name__}:{str(exc)[:500]}"
        rows.append(row)
        print(f"Reserve {priority}/3 {dataset_id}: {row['status']}")

    reserve_evidence = pd.DataFrame(rows)
    if TEST_MODE and os.environ.get("CDO_STAGE11C_TEST_INJECT_OUTCOME") == "1":
        reserve_evidence.loc[reserve_evidence.index[-1], "decision_basis"] = "source AUC selected this reserve"
    reserve_decisions = reserve_evidence[[
        "reserve_priority", "dataset_id", "metadata_gate_passed", "prefit_qualified",
        "activated", "status", "decision_basis", "performance_evaluated"
    ]].copy()
    write_immutable_csv(RESERVE_EVIDENCE_PATH, reserve_evidence)
    write_immutable_csv(RESERVE_DECISION_PATH, reserve_decisions)

assert reserve_evidence.sort_values("reserve_priority")["dataset_id"].tolist() == RESERVE_IDS
assert not bool_series(reserve_evidence, "performance_evaluated").any()
runtime_state.update({
    "reserves_screened": int(len(reserve_evidence)),
    "reserves_activated": int(bool_series(reserve_evidence, "activated").sum()),
    "last_updated_utc": utc_now(),
})
if not REPLAY_MODE:
    atomic_json(RUNTIME_STATE_PATH, runtime_state)
display(reserve_evidence[["reserve_priority", "dataset_id", "rights_verified", "patient_grouping_proven", "metadata_gate_passed", "activated", "status"]])


Reserve 1/3 UDIAT_B_2017: HOLD_REUSE_RIGHTS_NOT_VERIFIED
Reserve 2/3 MICCAI_BUV_2022: HOLD_COMPLETE_PATIENT_OR_LESION_MAP_NOT_PROVEN
Reserve 3/3 UVBLS200_2025: HOLD_REUSE_RIGHTS_NOT_VERIFIED


,reserve_priority,dataset_id,rights_verified,patient_grouping_proven,metadata_gate_passed,activated,status
0,1,UDIAT_B_2017,False,False,False,False,HOLD_REUSE_RIGHTS_NOT_VERIFIED
1,2,MICCAI_BUV_2022,True,False,False,False,HOLD_COMPLETE_PATIENT_OR_LESION_MAP_NOT_PROVEN
2,3,UVBLS200_2025,False,False,False,False,HOLD_REUSE_RIGHTS_NOT_VERIFIED


In [5]:
# @title 11C-4. Freeze the original-first development roster and exact Stage 11D handoff
if REPLAY_MODE:
    final_roster = pd.read_csv(ROSTER_PATH)
    handoff = verify_self_hashed_json(HANDOFF_PATH, "handoff_sha256")
    blind_stage11c = pd.read_csv(BLIND_CONTINUITY_PATH)
else:
    roster_rows = []
    for row in original_outcomes.sort_values("original_priority").to_dict("records"):
        if str(row.get("prefit_qualified", "")).lower() in {"true", "1"}:
            roster_rows.append({
                "roster_position": len(roster_rows) + 1, "dataset_id": row["dataset_id"],
                "roster_role": "ORIGINAL_DEVELOPMENT_EXTENSION", "frozen_origin_priority": int(row["original_priority"]),
                "activation_basis": "ORIGINAL_PASSED_COMPLETE_PREFIT_GOVERNANCE_GATES",
                "source_axis_fit_in_stage11c": False, "stage12_role_permitted": False,
            })
    for row in reserve_evidence.sort_values("reserve_priority").to_dict("records"):
        if str(row.get("activated", "")).lower() in {"true", "1"} and len(roster_rows) < TARGET_ROSTER:
            roster_rows.append({
                "roster_position": len(roster_rows) + 1, "dataset_id": row["dataset_id"],
                "roster_role": "A1_ACTIVATED_DEVELOPMENT_RESERVE", "frozen_origin_priority": int(row["reserve_priority"]),
                "activation_basis": row["decision_basis"], "source_axis_fit_in_stage11c": False,
                "stage12_role_permitted": False,
            })
    final_roster = pd.DataFrame(roster_rows, columns=[
        "roster_position", "dataset_id", "roster_role", "frozen_origin_priority",
        "activation_basis", "source_axis_fit_in_stage11c", "stage12_role_permitted"
    ])
    write_immutable_csv(ROSTER_PATH, final_roster)

    stage11d_authorised = len(final_roster) >= MINIMUM_FEASIBLE_DOMAINS
    decision = (
        "SEAL_STAGE11C_PREFIT_ROSTER_AUTHORISE_STAGE11D_FROZEN_ROSTER_SOURCE_RECOVERABILITY_KEEP_STAGE12_DDO2_AND_BLIND_PROHIBITED"
        if stage11d_authorised else
        "HOLD_STAGE11C_INSUFFICIENT_PREFIT_QUALIFIED_DOMAINS_KEEP_STAGE12_DDO2_AND_BLIND_PROHIBITED"
    )
    handoff = {
        "stage": "Stage11C", "handoff_target": "Stage11D",
        "decision": decision, "stage11c_protocol_seal_sha256": protocol_seal["seal_sha256"],
        "a1_final_record_sha256": a1_final["final_record_sha256"],
        "original_recovery_outcome_sha256": sha256_file(ORIGINAL_OUTCOME_PATH),
        "reserve_screening_evidence_sha256": sha256_file(RESERVE_EVIDENCE_PATH),
        "reserve_activation_decision_sha256": sha256_file(RESERVE_DECISION_PATH),
        "final_roster_sha256": sha256_file(ROSTER_PATH),
        "development_roster_ids": final_roster["dataset_id"].tolist(),
        "development_roster_count": int(len(final_roster)),
        "minimum_feasible_domains": MINIMUM_FEASIBLE_DOMAINS,
        "target_roster_count": TARGET_ROSTER,
        "stage11d_authorised": bool(stage11d_authorised),
        "stage11d_required_before_any_embedding": [
            "materialise every official pixel asset and verify provider or computed checksums",
            "verify released pathology mapping and patient/lesion group joins at image level",
            "deduplicate within and across the frozen development roster before splits",
            "seal grouped folds and held-out partitions before source-axis fitting",
        ],
        "stage11d_permitted": "frozen-roster development-only source recoverability and development-edge evaluation after all pre-embedding checks pass",
        "stage11d_prohibited": [
            "reserve replacement after any embedding or performance result", "Stage12", "DDO2 fitting",
            "locked-blind metadata, image or label access", "blind validation",
        ],
        "locked_blind_assets_touched": False,
        "embeddings_computed_in_stage11c": False,
        "source_axes_fitted_in_stage11c": False,
        "transfer_edges_evaluated_in_stage11c": False,
    }
    handoff["handoff_sha256"] = sha256_json(handoff)
    write_immutable_json(HANDOFF_PATH, handoff)

    blind_stage11c = blind_continuity.copy()
    blind_stage11c["stage11c_metadata_accessed"] = False
    blind_stage11c["stage11c_images_accessed"] = False
    blind_stage11c["stage11c_labels_accessed"] = False
    blind_stage11c["stage11c_development_role"] = False
    blind_stage11c["stage11c_status"] = "UNCHANGED_PERMANENTLY_LOCKED_BLIND"
    write_immutable_csv(BLIND_CONTINUITY_PATH, blind_stage11c)

assert final_roster["dataset_id"].is_unique
assert len(final_roster) <= TARGET_ROSTER
assert set(final_roster["dataset_id"]).isdisjoint(LOCKED_BLIND_IDS)
runtime_state.update({
    "final_development_roster_count": int(len(final_roster)),
    "stage11d_authorised": bool(handoff["stage11d_authorised"]),
    "stage12_authorised": False, "locked_blind_assets_touched": False,
    "last_updated_utc": utc_now(),
})
if not REPLAY_MODE:
    atomic_json(RUNTIME_STATE_PATH, runtime_state)
print("Frozen roster:", final_roster["dataset_id"].tolist())
print("Stage11D authorised:", handoff["stage11d_authorised"])


Frozen roster: []
Stage11D authorised: False


In [6]:
# @title 11C-5. Run independent validity, selection-bias, and locked-blind firewall checks
checks = []
def check(name, condition, evidence):
    checks.append({"check": name, "passed": bool(condition), "evidence": str(evidence)[:1200]})

check("A1 final record inherited", handoff["a1_final_record_sha256"] == a1_final["final_record_sha256"], handoff["a1_final_record_sha256"])
check("A1 protocol seal inherited", protocol_seal["parent_a1_protocol_seal_sha256"] == a1_seal["seal_sha256"], protocol_seal["parent_a1_protocol_seal_sha256"])
check("A1 route registry inherited", protocol_seal["a1_route_registry_sha256"] == sha256_file(A1_ROUTE_PATH), protocol_seal["a1_route_registry_sha256"])
check("A1 reserve pool inherited", protocol_seal["a1_reserve_pool_sha256"] == sha256_file(A1_RESERVE_PATH), protocol_seal["a1_reserve_pool_sha256"])
check("A1 activation policy inherited", protocol_seal["a1_activation_policy_sha256"] == activation_policy["policy_sha256"], protocol_seal["a1_activation_policy_sha256"])
check("five originals exact order", original_outcomes.sort_values("original_priority")["dataset_id"].tolist() == ORIGINAL_IDS, original_outcomes["dataset_id"].tolist())
check("three reserves exact order", reserve_evidence.sort_values("reserve_priority")["dataset_id"].tolist() == RESERVE_IDS, reserve_evidence["dataset_id"].tolist())
check("all originals adjudicated before reserves", bool_series(reserve_evidence, "screened_after_all_originals").all() and len(original_outcomes) == 5, "5 original outcomes precede reserve decisions")
check("original performance unused", not bool_series(original_outcomes, "performance_evaluated").any(), original_outcomes["performance_evaluated"].tolist())
check("reserve performance unused", not bool_series(reserve_evidence, "performance_evaluated").any(), reserve_evidence["performance_evaluated"].tolist())
selection_text = " ".join(reserve_evidence["decision_basis"].fillna("").astype(str)).lower()
check("no forbidden outcome activation input", not any(term in selection_text for term in FORBIDDEN_OUTCOME_TERMS), selection_text)
activated = reserve_evidence[bool_series(reserve_evidence, "activated")]
check("activated reserves pass metadata gate", len(activated) == 0 or bool_series(activated, "metadata_gate_passed").all(), activated["dataset_id"].tolist())
check("activated reserves are prefit qualified", len(activated) == 0 or bool_series(activated, "prefit_qualified").all(), activated["dataset_id"].tolist())
activated_order = activated.sort_values("reserve_priority")["dataset_id"].tolist()
expected_activated_order = [x for x in RESERVE_IDS if x in activated_order]
check("reserve activation respects frozen order", activated_order == expected_activated_order, activated_order)
originals_ready = original_outcomes[bool_series(original_outcomes, "prefit_qualified")].sort_values("original_priority")["dataset_id"].tolist()
expected_roster = (originals_ready + activated_order)[:TARGET_ROSTER]
check("original-first roster exact", final_roster["dataset_id"].tolist() == expected_roster, expected_roster)
check("roster has no explicit exclusion", set(final_roster["dataset_id"]).isdisjoint(set(exclusion_registry.iloc[:, 0].astype(str))), final_roster["dataset_id"].tolist())
check("roster has no locked-blind dataset", set(final_roster["dataset_id"]).isdisjoint(LOCKED_BLIND_IDS), final_roster["dataset_id"].tolist())
check("locked-blind count unchanged", blind_stage11c["dataset_id"].tolist() == LOCKED_BLIND_IDS, blind_stage11c["dataset_id"].tolist())
blind_access_columns = [x for x in blind_stage11c.columns if x.startswith("stage11c_") and x.endswith(("accessed", "role"))]
check("no locked-blind metadata image label or role", not any(bool_series(blind_stage11c, x).any() for x in blind_access_columns), blind_access_columns)
check("request log contains no blind request", len(request_audit) == 0 or not bool_series(request_audit, "locked_blind_request").any(), len(request_audit))
request_hosts_ok = True
for row in request_audit.to_dict("records"):
    host = str(row.get("resolved_host", "")).lower()
    allowed = permitted_hosts(str(row.get("dataset_id", "")))
    request_hosts_ok &= bool(host and (host in allowed or any(host.endswith("." + x) for x in allowed)))
check("every request resolved to an official host", request_hosts_ok, sorted(set(request_audit.get("resolved_host", pd.Series(dtype=str)).dropna().astype(str))))
check("roster size capped at target", len(final_roster) <= TARGET_ROSTER, len(final_roster))
check("Stage11D count rule exact", bool(handoff["stage11d_authorised"]) == (len(final_roster) >= MINIMUM_FEASIBLE_DOMAINS), len(final_roster))
check("roster sealed before any modelling", not bool_series(final_roster, "source_axis_fit_in_stage11c").any(), "all false")
check("no embedding computed", runtime_state["embeddings_computed"] is False, runtime_state["embeddings_computed"])
check("no source axis fitted", runtime_state["source_axes_fitted"] is False, runtime_state["source_axes_fitted"])
check("no transfer edge evaluated", runtime_state["transfer_edges_evaluated"] is False, runtime_state["transfer_edges_evaluated"])
check("DDO2 not fitted", runtime_state["ddo2_fitted"] is False, runtime_state["ddo2_fitted"])
check("Stage12 remains prohibited", runtime_state["stage12_authorised"] is False and not bool_series(final_roster, "stage12_role_permitted").any(), "false")
check("blind validation not performed", runtime_state["blind_validation_performed"] is False, "false")
check("handoff self hash valid", sha256_json({k: v for k, v in handoff.items() if k != "handoff_sha256"}) == handoff["handoff_sha256"], handoff["handoff_sha256"])

validity = pd.DataFrame(checks)
if REPLAY_MODE:
    prior_validity = pd.read_csv(VALIDITY_PATH)
    assert canonical_csv_text(prior_validity) == canonical_csv_text(validity), "Replay validity table changed"
else:
    write_immutable_csv(VALIDITY_PATH, validity)
failed = validity.loc[~validity["passed"].astype(bool), "check"].tolist()
assert not failed, "Independent Stage11C validity checks failed: " + "; ".join(failed)
print(f"Independent validity checks: {int(validity['passed'].sum())}/{len(validity)} passed")


Independent validity checks: 31/31 passed


In [7]:
# @title 11C-6. Seal report, integrity manifest, final record, and Replay invariance
import matplotlib.pyplot as plt

original_ready = int(bool_series(original_outcomes, "prefit_qualified").sum())
original_routes = int(bool_series(original_outcomes, "official_route_recovered").sum())
reserves_activated = int(bool_series(reserve_evidence, "activated").sum())
roster_count = int(len(final_roster))
stage11d_authorised = bool(handoff["stage11d_authorised"])
decision = handoff["decision"]
next_step = (
    "BUILD_STAGE11D_FROZEN_ROSTER_SOURCE_RECOVERABILITY_AND_DEVELOPMENT_EDGE_EVALUATION"
    if stage11d_authorised else
    "REVIEW_STAGE11C_TYPED_HOLDS_AND_MANUAL_OFFICIAL_RECEIPT_OPTIONS_WITHOUT_BLIND_ACCESS_OR_PERFORMANCE_BASED_SUBSTITUTION"
)

report = f"""# Amended Stage 11C report\n\n## Answer first\n\n- Original official routes recovered: **{original_routes}/5**.\n- Original pre-fit-qualified domains: **{original_ready}/5**.\n- Reserves screened / activated: **{len(reserve_evidence)}/{reserves_activated}**.\n- Final frozen development roster: **{roster_count}**; minimum feasible: **{MINIMUM_FEASIBLE_DOMAINS}**.\n- Stage 11D authorised: **{stage11d_authorised}**.\n- Decision: `{decision}`.\n\n## Method boundary\n\nThe roster used only official identity, rights, released pathology semantics, patient/lesion grouping, integrity, and development-only provenance evidence. No embedding or outcome entered reserve activation. No source axis, transfer edge, DDO2 coefficient, Stage 12 operation, locked-blind access, or blind validation occurred.\n\n## Frozen roster\n\n{final_roster.fillna('').to_markdown(index=False) if len(final_roster) else 'No dataset reached the complete pre-fit handoff gate.'}\n\n## Typed holds\n\n### Originals\n\n{original_outcomes[['dataset_id','status','hold_reason']].fillna('').to_markdown(index=False)}\n\n### Reserves\n\n{reserve_evidence[['dataset_id','status','decision_basis']].fillna('').to_markdown(index=False)}\n"""
if REPLAY_MODE:
    assert REPORT_PATH.read_text(encoding="utf-8") == report
else:
    write_immutable_text(REPORT_PATH, report)

if not REPLAY_MODE:
    plot_rows = []
    for row in original_outcomes.to_dict("records"):
        plot_rows.append((row["dataset_id"], "Original", 2 if str(row["prefit_qualified"]).lower() in {"true", "1"} else (1 if str(row["official_route_recovered"]).lower() in {"true", "1"} else 0)))
    for row in reserve_evidence.to_dict("records"):
        plot_rows.append((row["dataset_id"], "Reserve", 2 if str(row["activated"]).lower() in {"true", "1"} else (1 if str(row["metadata_gate_passed"]).lower() in {"true", "1"} else 0)))
    fig, ax = plt.subplots(figsize=(10.5, 5.2))
    labels = [x[0] for x in plot_rows]
    values = [x[2] for x in plot_rows]
    colors = ["#1976D2" if x[1] == "Original" else "#8E5CC7" for x in plot_rows]
    ax.barh(range(len(labels)), values, color=colors, alpha=.9)
    ax.set_yticks(range(len(labels)), labels)
    ax.set_xticks([0, 1, 2], ["Hold", "Metadata/route", "Pre-fit qualified"])
    ax.set_xlim(-0.05, 2.15); ax.invert_yaxis(); ax.grid(axis="x", alpha=.2)
    ax.set_title("Stage 11C — outcome-free original and reserve adjudication")
    fig.tight_layout(); fig.savefig(FIGURE_PATH, dpi=180, bbox_inches="tight"); plt.close(fig)

tracked_paths = [
    PROTOCOL_SEAL_PATH, PARENT_COMMITMENT_PATH, REQUEST_LOG_PATH, FILE_RECEIPT_PATH,
    ORIGINAL_OUTCOME_PATH, RESERVE_EVIDENCE_PATH, RESERVE_DECISION_PATH, ROSTER_PATH,
    HANDOFF_PATH, BLIND_CONTINUITY_PATH, VALIDITY_PATH, REPORT_PATH, FIGURE_PATH,
]
manifest = pd.DataFrame([{
    "relative_path": str(path.relative_to(STAGE11C_ROOT)), "size_bytes": path.stat().st_size,
    "sha256": sha256_file(path)
} for path in tracked_paths])
if REPLAY_MODE:
    prior_manifest = pd.read_csv(OUTPUT_MANIFEST_PATH)
    assert canonical_csv_text(prior_manifest) == canonical_csv_text(manifest), "Replay output manifest changed"
    for row in prior_manifest.to_dict("records"):
        path = STAGE11C_ROOT / row["relative_path"]
        assert path.is_file() and path.stat().st_size == int(row["size_bytes"]) and sha256_file(path) == row["sha256"]
else:
    write_immutable_csv(OUTPUT_MANIFEST_PATH, manifest)

final_payload = {
    "stage": "AmendedStage11C", "protocol_version": "0.1", "decision": decision,
    "a1_final_record_sha256": a1_final["final_record_sha256"],
    "stage11c_protocol_seal_sha256": protocol_seal["seal_sha256"],
    "official_route_registry_sha256": sha256_file(A1_ROUTE_PATH),
    "original_recovery_outcome_sha256": sha256_file(ORIGINAL_OUTCOME_PATH),
    "reserve_screening_evidence_sha256": sha256_file(RESERVE_EVIDENCE_PATH),
    "reserve_activation_decision_sha256": sha256_file(RESERVE_DECISION_PATH),
    "final_roster_sha256": sha256_file(ROSTER_PATH),
    "stage11d_handoff_sha256": handoff["handoff_sha256"],
    "locked_blind_continuity_sha256": sha256_file(BLIND_CONTINUITY_PATH),
    "independent_validity_sha256": sha256_file(VALIDITY_PATH),
    "output_integrity_manifest_sha256": sha256_file(OUTPUT_MANIFEST_PATH),
    "original_official_routes_recovered": original_routes,
    "original_prefit_qualified_domains": original_ready,
    "reserves_screened": int(len(reserve_evidence)), "reserves_activated": reserves_activated,
    "final_development_roster_count": roster_count, "minimum_feasible_domains": MINIMUM_FEASIBLE_DOMAINS,
    "stage11d_authorised": stage11d_authorised, "stage12_authorised": False,
    "ddo2_fitted": False, "locked_blind_assets_touched": False, "blind_validation_performed": False,
    "next_step": next_step,
}
if REPLAY_MODE:
    final_record = verify_self_hashed_json(FINAL_RECORD_PATH, "final_record_sha256")
    for key, value in final_payload.items():
        assert final_record[key] == value, f"Replay final record changed at {key}"
else:
    final_record = dict(final_payload); final_record["completed_utc"] = utc_now()
    final_record["final_record_sha256"] = sha256_json(final_record)
    write_immutable_json(FINAL_RECORD_PATH, final_record)

runtime_state.update({"completed": True, "decision": decision, "final_record_sha256": final_record["final_record_sha256"], "last_updated_utc": utc_now()})
atomic_json(RUNTIME_STATE_PATH, runtime_state)

# Raw provider assets stay in ephemeral /content only. Removing them after all evidence is sealed
# prevents accidental Drive persistence and makes Replay provider-free.
if not REPLAY_MODE and TEMP_ROOT.exists():
    shutil.rmtree(TEMP_ROOT)

print("================ AMENDED STAGE 11C COMPLETE ================")
print("Original official routes recovered:", f"{original_routes}/5")
print("Original prefit-qualified domains:", f"{original_ready}/5")
print("Reserves screened / activated:", f"{len(reserve_evidence)}/{reserves_activated}")
print("Final development roster / minimum feasible:", f"{roster_count}/{MINIMUM_FEASIBLE_DOMAINS}")
print("Stage11D authorised:", stage11d_authorised)
print("Decision:", decision)
print("Protocol seal:", protocol_seal["seal_sha256"])
print("Original recovery outcome hash:", final_record["original_recovery_outcome_sha256"])
print("Reserve screening evidence hash:", final_record["reserve_screening_evidence_sha256"])
print("Final roster hash:", final_record["final_roster_sha256"])
print("Stage11D handoff hash:", final_record["stage11d_handoff_sha256"])
print("Final record hash:", final_record["final_record_sha256"])
print("Next step:", next_step)


================ AMENDED STAGE 11C COMPLETE ================
Original official routes recovered: 1/5
Original prefit-qualified domains: 0/5
Reserves screened / activated: 3/0
Final development roster / minimum feasible: 0/4
Stage11D authorised: False
Decision: HOLD_STAGE11C_INSUFFICIENT_PREFIT_QUALIFIED_DOMAINS_KEEP_STAGE12_DDO2_AND_BLIND_PROHIBITED
Protocol seal: f708e91420a13afc6dc9791131b8471b45e9eab38ca73e90c42d85f2f378ddab
Original recovery outcome hash: 24393b432df84d9cd61d53576462e15735dbaee961953dd576ea30e074ddecda
Reserve screening evidence hash: 16abcb0bbc8625938e74be8e1983b27f5439a35b6c951c373d879b4f67e11721
Final roster hash: 753d2cc5b14b7b4e555ca22845e0e208a783bdbabe5e49b9fafa8705557a160e
Stage11D handoff hash: 7e01584660cd91cd93077f2c7d67b822ba9311cb94b29159b8b4647aed7074bc
Final record hash: 05e0288a02af5255ce0335d6da74f78483c51ee8c9f2b63de60a6dd2a6fb1ada
Next step: REVIEW_STAGE11C_TYPED_HOLDS_AND_MANUAL_OFFICIAL_RECEIPT_OPTIONS_WITHOUT_BLIND_ACCESS_OR_PERFORMANCE_BASED_